# Sensitivity Analysis

This notebook performs sensitivity analysis on the ABM model using the best parameters found from optimization.

**Analyses:**
1.  **Time Horizon:** 150-year simulation, recording objective score every 10 years.
2.  **Number of Agents:** Varying agents (5, 15, 20).
3.  **Territory Flexibility:** Fixed vs. Flexible territory.

In [ ]:
import sensitivity_model
import optuna
import numpy as np
import pandas as pd
import os
import geopandas as gpd
import matplotlib.pyplot as plt
import random
import seaborn as sns

# 1. Load Best Parameters
study = optuna.load_study(study_name='opt_2_26_v1', storage=r"sqlite:///D:\itay\ABM\Results\run\opt_run.db")
best_params = study.best_params
print("Best Params:", best_params)

# Normalize weights
sum_w = sum(best_params.values())
norm_params = {k: v / sum_w for k, v in best_params.items()}

permanent_weights_dict = {
    "dist_to_kb": norm_params["dist_to_kb"],
    "p_water": norm_params["p_water"],
    "Mean_rain": norm_params["Mean_rain"],
    "slope_suitability": norm_params["slope_suitability"],
}

yearly_weights_dict = {
    "return_to_site": norm_params["return_to_site"],
    "humen_stress": norm_params["humen_stress"],
    "Yearly_rain": norm_params["Yearly_rain"],
}

ras_wts = list(permanent_weights_dict.values()) + list(yearly_weights_dict.values())

# Load Calibration Data
gdf_real = gpd.read_file(sensitivity_model.CALIB_SHP_PATH)

# Ensure Data Loaded
sensitivity_model.ensure_data_loaded()

## 1. Time Horizon Analysis (150 Years)

In [ ]:
def run_long_simulation(n_years=150, check_interval=10, seed=42):
    print(f"Running {n_years} year simulation...")
    
    # Setup directories
    base_dir = r"D:\itay\ABM\Results\sensitivity\time_horizon"
    os.makedirs(base_dir, exist_ok=True)
    
    # Initialize Model
    if seed is not None:
        random.seed(seed)
        np.random.seed(seed)

    y_output_data = sensitivity_model.GlobalData.y_output
    inds = list(range(len(y_output_data)))
    random.shuffle(inds)

    suitability_raster, return_raster, stress_ras = sensitivity_model.get_suitability_raster(
        y_output_data, inds, year=0, weights=ras_wts
    )

    model = sensitivity_model.NomadModel(suitability_raster, return_raster, stress_ras, inds, ras_w=ras_wts, y_output=y_output_data)
    
    results = []

    for i in range(n_years):
        model.step()
        
        # Checkpoint every 10 years
        if (i + 1) % check_interval == 0:
            print(f"  Evaluating at year {i+1}...")
            step_dir = os.path.join(base_dir, f"year_{i+1}")
            os.makedirs(step_dir, exist_ok=True)
            
            gdf_model = sensitivity_model.to_gdf(model)
            total_error, ratio_error = sensitivity_model.obj_func(gdf_real, gdf_model, step_dir)
            
            # Calculate spatial error from total and ratio: total = (spatial + ratio) / 2
            spatial_error = (total_error * 2) - ratio_error
            
            results.append({
                "Year": i + 1,
                "Total_Score": total_error,
                "Spatial_Error": spatial_error,
                "Ratio_Error": ratio_error
            })

        if i != (n_years - 1):
            model.move_year(inds)
            
    return pd.DataFrame(results)

df_time = run_long_simulation(n_years=150, check_interval=10)
print(df_time)

# Plot
plt.figure(figsize=(10, 6))
plt.plot(df_time['Year'], df_time['Total_Score'], label='Total Score (Obj Func)', marker='o')
plt.plot(df_time['Year'], df_time['Spatial_Error'], label='Spatial Error', marker='s')
plt.plot(df_time['Year'], df_time['Ratio_Error'], label='Ratio Error', marker='^')
plt.xlabel('Year')
plt.ylabel('Error (Lower is better)')
plt.title('Objective Function over 150 Years')
plt.legend()
plt.grid(True)
plt.show()

## 2. Number of Agents Analysis

In [ ]:
def run_agent_sensitivity(agent_counts=[5, 15, 20], n_iterations=5):
    results = []
    base_dir = r"D:\itay\ABM\Results\sensitivity\agents"
    
    for n_agents in agent_counts:
        print(f"Testing with {n_agents} agents...")
        scores = []
        for i in range(n_iterations):
            seed = 1000 + (n_agents * 100) + i
            
            # Run model using run_model_opt wrapper
            model, run_dir = sensitivity_model.run_model_opt(
                model_years=75,
                ras_wts=ras_wts,
                main_run_directory=base_dir,
                trial_number=f"agents_{n_agents}",
                iteration=i,
                num_agents=n_agents,
                seed=seed
            )
            
            gdf_model = sensitivity_model.to_gdf(model)
            score, ratio = sensitivity_model.obj_func(gdf_real, gdf_model, run_dir)
            scores.append(score)
            
            results.append({
                "Num_Agents": n_agents,
                "Iteration": i,
                "Score": score
            })
            
    return pd.DataFrame(results)

df_agents = run_agent_sensitivity()

# Plot
plt.figure(figsize=(8, 6))
sns.boxplot(x='Num_Agents', y='Score', data=df_agents)
plt.title('Sensitivity to Number of Agents')
plt.show()

## 3. Territory Flexibility Analysis

In [ ]:
def run_territory_sensitivity(n_iterations=5):
    results = []
    base_dir = r"D:\itay\ABM\Results\sensitivity\territory"
    
    # We compare Fixed vs Flexible (Default)
    settings = {"Flexible": False, "Fixed": True}
    
    for name, is_fixed in settings.items():
        print(f"Testing {name} Territory...")
        for i in range(n_iterations):
            seed = 2000 + (int(is_fixed) * 100) + i
            
            model, run_dir = sensitivity_model.run_model_opt(
                model_years=75,
                ras_wts=ras_wts,
                main_run_directory=base_dir,
                trial_number=f"fixed_{is_fixed}",
                iteration=i,
                fixed_territory=is_fixed,
                seed=seed
            )
            
            gdf_model = sensitivity_model.to_gdf(model)
            score, ratio = sensitivity_model.obj_func(gdf_real, gdf_model, run_dir)
            
            results.append({
                "Territory_Type": name,
                "Iteration": i,
                "Score": score
            })
            
    return pd.DataFrame(results)

df_territory = run_territory_sensitivity()

# Plot
plt.figure(figsize=(8, 6))
sns.boxplot(x='Territory_Type', y='Score', data=df_territory)
plt.title('Sensitivity to Territory Flexibility')
plt.show()

## 4. Null Weight Baseline Comparison

This test establishes a baseline by running the model with equal weights for all 7 parameters (1/7 each). It compares the optimized model's performance against a model with no preference structure.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

def run_null_baseline(n_iterations=30):
    print(f"Running Null Weight Baseline ({n_iterations} iterations)...")

    # Equal weights (1/7)
    val = 1.0 / 7.0
    null_perm_weights = {
        "dist_to_kb": val,
        "p_water": val,
        "Mean_rain": val,
        "slope_suitability": val,
    }
    null_yearly_weights = {
        "return_to_site": val,
        "humen_stress": val,
        "Yearly_rain": val,
    }

    # Combined list for raster calculation
    null_ras_wts = list(null_perm_weights.values()) + list(null_yearly_weights.values())

    base_dir = r"D:\itay\ABM\Results\sensitivity\null_baseline"
    scores = []

    for i in range(n_iterations):
        seed = 3000 + i

        model, run_dir = sensitivity_model.run_model_opt(
            model_years=75,
            ras_wts=null_ras_wts,
            main_run_directory=base_dir,
            trial_number="null_baseline",
            iteration=i,
            permanent_weights_dict=null_perm_weights,
            yearly_weights_dict=null_yearly_weights,
            seed=seed
        )

        gdf_model = sensitivity_model.to_gdf(model)
        score, ratio = sensitivity_model.obj_func(gdf_real, gdf_model, run_dir)
        scores.append(score)

    return scores

# Execute the baseline run
null_scores = run_null_baseline(n_iterations=30)

# Statistics
null_mean = np.mean(null_scores)
null_std = np.std(null_scores)
null_ci95 = 1.96 * (null_std / np.sqrt(len(null_scores)))

print("\n--- Null Baseline Results ---")
print(f"Mean Score: {null_mean:.4f}")
print(f"95% CI: [{null_mean - null_ci95:.4f}, {null_mean + null_ci95:.4f}]")

# Comparison with Best Trial
# Assuming 'study' is your optuna study object
best_trial_score = study.best_value
print(f"Best Optimized Score: {best_trial_score:.4f}")

if best_trial_score < (null_mean - null_ci95):
    print("RESULT: Optimization significantly improved upon random weights.")
else:
    print("RESULT: Optimization did not significantly outperform random weights.")

# Visualization
plt.figure(figsize=(8, 6))
sns.histplot(null_scores, kde=True, color='gray', label='Null Baseline')
plt.axvline(best_trial_score, color='red', linestyle='--', label=f'Best Optimized ({best_trial_score:.4f})')
plt.axvline(null_mean, color='black', linestyle='-', label=f'Null Mean ({null_mean:.4f})')
plt.xlabel('Objective Score (Error)')
plt.title('Null Weight Baseline vs. Optimized Model')
plt.legend()
plt.show()

# Export Results

In [ ]:
# Export all dataframes to CSV
print("Exporting results to CSV...")

# 1. Time Horizon
if 'df_time' in locals():
    path = r"D:\itay\ABM\Results\sensitivity\time_horizon\time_errors.csv"
    df_time.to_csv(path, index=False)
    print(f"Saved: {path}")

# 2. Agents Sensitivity
if 'df_agents' in locals():
    path = r"D:\itay\ABM\Results\sensitivity\agents\agents_sensitivity.csv"
    df_agents.to_csv(path, index=False)
    print(f"Saved: {path}")

# 3. Territory Sensitivity
if 'df_territory' in locals():
    path = r"D:\itay\ABM\Results\sensitivity\territory\territory_sensitivity.csv"
    df_territory.to_csv(path, index=False)
    print(f"Saved: {path}")

# 4. Null Baseline
if 'null_scores' in locals():
    path = r"D:\itay\ABM\Results\sensitivity\null_baseline\null_baseline_scores.csv"
    pd.DataFrame(null_scores, columns=["Score"]).to_csv(path, index=False)
    print(f"Saved: {path}")